In [29]:
from boson_data_lib import *
import numpy as np
import scipy.linalg as sl
import os, glob
import pandas as pd

In [30]:
data_dir  = "../DATA/"
tests_dir = "../TESTS/"

In [31]:
# Fidelity distance of each test state from the "best" reference initial state
best_init_rho = np.array([[ 0.96693975+0.j        , -0.17832301+0.00560344j],
                           [-0.17832301-0.00560344j,  0.03306025+0.j        ]])

fid_dists = fidelity_distances(data_dir, best_init_rho)

In [32]:
def read_nonmark_dataframe(filename, fid_distances, data_dir, tests_dir):
    """Read a NonMark HDF5 test file and return a tidy DataFrame."""
    df = pd.DataFrame()
    print("Processing", filename, "...")

    with h5py.File(tests_dir + filename, "r") as f:
        for gamma in f.keys():
            g = gamma[6:]          # strip "gamma_" prefix
            print(" ", gamma)

            init_states = ["State_D" + str(i) for i in range(1, 21)]

            for state in init_states:
                if state not in f[gamma]:
                    continue       # skip if fidelity not stored (incomplete run)

                Fidelity = f[gamma][state]["Fidelity"][()]
                ser_len  = len(Fidelity)

                f_name = data_dir + state + "_2CUT_data.h5"
                t, dt  = extract_time(f_name, g)
                t = t[:ser_len]    # align lengths

                df = pd.concat([df, pd.DataFrame({
                    "Gamma"      : [g]              * ser_len,
                    "State"      : [state[7:]]      * ser_len,
                    "Time"       : t,
                    "gt"         : np.array(t) * float(g),
                    "Fidelity"   : Fidelity,
                    "Infidelity" : 1 - Fidelity,
                    "Distance"   : [fid_distances[int(state[7:]) - 1]] * ser_len,
                })])

    print("done!")
    return df

In [33]:
# List available NonMark result files
nonmark_files = sorted(glob.glob(tests_dir + "NonMark_*.h5"))
for f in nonmark_files:
    print(os.path.basename(f))

NonMark_Kossak2H_k5_trn4_tst20_2026-Mar-12_at_20-40.h5
NonMark_Kossak_k10_trn4_tst20_2026-Apr-01_at_22-02.h5
NonMark_Kossak_k1_trn4_tst20_2026-Apr-01_at_22-29.h5
NonMark_Kossak_k1_trn4_tst20_2026-Mar-12_at_18-53.h5
NonMark_Kossak_k5_trn4_tst20_2026-Apr-01_at_21-12.h5
NonMark_Kossak_k5_trn4_tst20_2026-Apr-01_at_21-45.h5
NonMark_Kossak_k5_trn4_tst20_2026-Mar-12_at_18-31.h5
NonMark_k5_trn4_tst20_2026-Mar-08_at_14-39.h5
NonMark_k5_trn4_tst20_2026-Mar-08_at_14-43.h5
NonMark_k5_trn4_tst20_2026-Mar-08_at_15-20.h5
NonMark_k5_trn4_tst20_2026-Mar-08_at_15-34.h5
NonMark_k5_trn4_tst20_2026-Mar-08_at_15-35.h5
NonMark_k5_trn4_tst20_2026-Mar-08_at_15-37.h5
NonMark_sepH_k5_trn4_tst20_2026-Mar-08_at_16-20.h5
NonMark_sharedH_k5_trn4_tst20_2026-Mar-08_at_16-13.h5
NonMark_sharedH_k5_trn4_tst20_2026-Mar-08_at_16-15.h5


In [35]:
# Pick the file to process (change index or set manually)
#test_file = os.path.basename("NonMark_Kossak_k5_trn4_tst20_2026-Apr-01_at_21-45.h5")
#test_file = os.path.basename("NonMark_Kossak_k10_trn4_tst20_2026-Apr-01_at_22-02.h5")
test_file = os.path.basename("NonMark_Kossak_k1_trn4_tst20_2026-Apr-01_at_22-29.h5")

print("Using:", test_file)

Using: NonMark_Kossak_k1_trn4_tst20_2026-Apr-01_at_22-29.h5


test_file ="NonMark_Kossak_k5_trn4_tst20_2026-Apr-01_at_21-45.h5"

In [36]:
df = read_nonmark_dataframe(test_file, fid_dists, data_dir, tests_dir)

Processing NonMark_Kossak_k1_trn4_tst20_2026-Apr-01_at_22-29.h5 ...
  gamma_0.079477
  gamma_0.25133
  gamma_0.79477
  gamma_2.5133
  gamma_25.133
  gamma_251.33
  gamma_7.9477
  gamma_79.477
done!


In [37]:
df

,Gamma,State,Time,gt,Fidelity,Infidelity,Distance
0,0.079477,1,0.06000,0.004769,1.000000,3.508305e-14,0.341926
1,0.079477,1,0.08000,0.006358,0.991805,8.194955e-03,0.341926
2,0.079477,1,0.10000,0.007948,0.983446,1.655373e-02,0.341926
3,0.079477,1,0.12000,0.009537,0.975277,2.472308e-02,0.341926
4,0.079477,1,0.14000,0.011127,0.966726,3.327399e-02,0.341926
...,...,...,...,...,...,...,...
751,79.477,20,0.23800,18.915526,0.999702,2.977224e-04,0.340014
752,79.477,20,0.23825,18.935395,0.999701,2.989793e-04,0.340014
753,79.477,20,0.23850,18.955264,0.999700,3.002304e-04,0.340014
754,79.477,20,0.23875,18.975134,0.999699,3.014750e-04,0.340014


In [38]:
#pkl_name = "dataframe_" + test_file[:-3] + ".pkl"

#pl_name = "dataframe_" + "NonMark_Kossak_k10_trn4_tst20_2026-Apr-01_at_22-02"+ ".pkl"
#pkl_name = "dataframe_" + "NonMark_Kossak_k10_trn4_tst20_2026-Apr-01_at_22-02.pkl"

pkl_name = "dataframe_" + "NonMark_Kossak_k1_trn4_tst20_2026-Apr-01_at_22-29.pkl"

df.to_pickle(tests_dir + pkl_name)
print("Saved:", tests_dir + pkl_name)

Saved: ../TESTS/dataframe_NonMark_Kossak_k1_trn4_tst20_2026-Apr-01_at_22-29.pkl
